In [ ]:
%config InlineBackend.figure_format = "retina"
from matplotlib import rcParams

rcParams["savefig.dpi"] = 100
rcParams["figure.dpi"] = 100
rcParams["font.size"] = 14

# LaTeX rendering
rcParams["text.usetex"] = True
rcParams["font.family"] = "serif"
rcParams["font.serif"] = ["Computer Modern"]

# Ticks — less relevant for maps but good to have for any non-map subplots
rcParams["xtick.direction"] = "in"
rcParams["ytick.direction"] = "in"
rcParams["xtick.top"] = True
rcParams["ytick.right"] = True
rcParams["xtick.minor.visible"] = True
rcParams["ytick.minor.visible"] = True

rcParams["axes.linewidth"] = 1.2
rcParams["lines.linewidth"] = 1.5
rcParams["legend.framealpha"] = 0.9
rcParams["legend.fontsize"] = 12

# Maps tend to look better with a slightly larger default figure size
rcParams["figure.figsize"] = (10, 6)

In [ ]:
import os
import sys

sys.path.insert(0, "/home/ssage/Karmma/KaRMMa_dg_RS")
import h5py as h5
import jax
import numpy as np

jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp

from karmma import KarmmaConfig, KarmmaSampler
from karmma.structs import ThetaParams, XlmParams
from karmma.utils.plotting_utils import plot_dm_comparison

In [ ]:
configfile = "/spiff/ssage/karmma_data/LN_dg_RS_linmu/Inputs/desy3.yaml"
config = KarmmaConfig(configfile)

analysis = config.analysis
io = config.io
mcmc = config.mcmc
nside = analysis.nside
nbins = analysis.nbins
mask = io.mask
io_dir = io.io_dir
n_samples = mcmc.n_samples

sampler = KarmmaSampler(
    dg_obs=io.dg_obs,
    mask=mask,
    CL=analysis.cl,
    alpha=analysis.alpha,
    beta=analysis.beta,
    N_bar=io.N_bar,
    infer_theta=False,
    theta_fixed=ThetaParams(
        **{field: np.zeros(nbins) for field in ThetaParams._fields}
    ),
    pixwin=analysis.pixwin,
)
x2dm_jit = jax.jit(sampler.x2dm)

with h5.File(os.path.join(io_dir, "Inputs/mock_dg.h5"), "r") as f:
    xlm_true = XlmParams(
        real=jnp.array(f["true_xlm/real"][:]), imag=jnp.array(f["true_xlm/imag"][:])
    )

dm_true = np.array(x2dm_jit(xlm_true))
print(
    f"dm_true shape: {dm_true.shape},  range: [{dm_true.min():.3f}, {dm_true.max():.3f}]"
)

In [ ]:
samples_path = os.path.join(io_dir, "samples.h5")

with h5.File(samples_path, "r") as f:
    dm_stored = "dm" in f
    n_stored = f["xlm/real"].shape[0]

if dm_stored:
    print(f"dm already in samples.h5 ({n_stored} samples), skipping reconstruction.")
else:
    print(f"Reconstructing dm for {n_stored} samples and storing in samples.h5...")
    with h5.File(samples_path, "r") as f:
        xlm_real = f["xlm/real"][:]
        xlm_imag = f["xlm/imag"][:]

    n_pix = sampler.map_shape[-1]
    dm_shape = (n_stored, nbins, n_pix)

    with h5.File(samples_path, "a") as f:
        ds = f.create_dataset("dm", shape=dm_shape, dtype=np.float64)
        for idx in range(n_stored):
            if idx % 20 == 0:
                print(f"  {idx}/{n_stored}")
            xlm_s = XlmParams(
                real=jnp.array(xlm_real[idx]), imag=jnp.array(xlm_imag[idx])
            )
            ds[idx] = np.array(x2dm_jit(xlm_s))

    print("Done.")

In [ ]:
print("Computing sample-mean dm map...")
with h5.File(os.path.join(io_dir, "samples.h5"), "r") as f:
    n_samples_loaded = f["dm"].shape[0]
    dm_mean = np.zeros_like(dm_true)
    for idx in range(n_samples_loaded):
        if idx % 20 == 0:
            print(f"  {idx}/{n_samples_loaded}")
        dm_mean += f["dm"][idx]

dm_mean /= n_samples_loaded
print(f"Done.  dm_mean range: [{dm_mean.min():.3f}, {dm_mean.max():.3f}]")

In [ ]:
plot_dm_comparison(dm_true, dm_mean, mask, n_samples=n_samples_loaded)